In [12]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import os
import json

load_dotenv(override=True)


True

In [4]:
todos = [] 
completions = []

In [7]:
def show(message):
    try:
        console = Console()
        console.print(message)
    except Exception as e:
        print(f"Error while showing message: {e}")

In [2]:
def add_todo(item: str):
    if not isinstance(item, str):
        raise TypeError("item must be a string")
    todos.append(item)

def complete_todo(index: int, completion_notes: str):
    if not isinstance(index, int):
        raise TypeError("index must be an integer")
    if index < 0 or index >= len(todos):
        raise IndexError("todo index out of range")

    if len(completions) < len(todos):
        completions.extend([False] * (len(todos) - len(completions)))

    completions[index] = True
    show(f"Completed todo: {todos[index]} - Notes: {completion_notes}")

def show_todos():
    if not todos:
        show("No todos yet.")
        return

    lines = []
    for i, item in enumerate(todos):
        done = i < len(completions) and completions[i]
        entry = f"{i}. {item}"
        if done:
            entry = f"[strike]{entry}[/strike]"
        lines.append(entry)

    show("\n".join(lines))
   

In [5]:
add_todo("Buy groceries") 
add_todo("Walk the dog")
add_todo("Read a book") 

In [9]:
show_todos()

0. Buy groceries
1. Walk the dog
2. Read a book

In [15]:
complete_todo(1, "Walked the dog in the park.")
show_todos()

Completed todo: Walk the dog - Notes: Walked the dog in the park.

0. Buy groceries
1. Walk the dog
2. Read a book

In [25]:
def handle_tools(tool_calls):
    results=[]
    for tool_call in tool_calls:
        show(f"calling tool: {tool_call.function.name} with args {tool_call.function.arguments}")
        tool = globals().get(tool_call.function.name)
        tool_args = json.loads(tool_call.function.arguments)
        result = tool(**tool_args) if tool else None
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [34]:
ollama_key = os.getenv('OLLAMA_API_KEY')
ollama = OpenAI(base_url='http://192.168.1.190:11434/v1', api_key=ollama_key)
model_name = "gpt-oss:20b-cloud"



In [28]:
tools = [
{
        "type": "function",
        "function": {
            "name": "add_todo",
            "description": "Add a new todo item to the todo list.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item": {
                        "type": "string",
                        "description": "The todo item text to add."
                    }
                },
                "required": ["item"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "complete_todo",
            "description": "Mark a todo item as completed and include completion notes.",
            "parameters": {
                "type": "object",
                "properties": {
                    "index": {
                        "type": "integer",
                        "description": "Zero-based index of the todo item to mark as completed."
                    },
                    "completion_notes": {
                        "type": "string",
                        "description": "Notes about how or when the todo was completed."
                    }
                },
                "required": ["index", "completion_notes"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "show_todos",
            "description": "Display the current todo list, with completed items marked.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
                "additionalProperties": False
            }
        }
    }
]



In [20]:
def loop(messages, max_iterations=5):
    done = False
    while not done:
        response = ollama.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=tools,
            reasoning_effort="none"
        )
        finish_reason = response.choices[0].finish_reason
        if(finish_reason == "tool_calls"):
            tool_calls = response.choices[0].message.tool_calls
            tool_results = handle_tools(tool_calls)
            messages.append(response.choices[0].message)
            messages.extend(tool_results)
        else:
            done = True
    show(f"AI: {response.choices[0].message.content}") 

In [41]:
system_prompt = """You are a mathematical genius. Explain your solution so that a 8 year old can understand. \
            Beak it down in steps. Use simple examples when possible. \
            You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn. \
            Now use the todo list tools, create a plan, carry out the steps, and reply with the solution. \
            Do not give away the ansewer directly. \
            Make them think and possibly hint at the final answer. \
            Do not make your response too long as they may lose interest. keep it brief. \
            Make the answer fun and relatable to Minecraft player (especially in creative mode)."""
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "How do we measure a qualdrilateral area?"}
]

todos = []
completions = []
answer = loop(messages)

print(answer)

calling tool: add_todo with args {"item":"Draw the quadrilateral on a big piece of paper so you can see all 
sides."}

calling tool: add_todo with args {"item":"Check if your shape looks like a rectangle or a square—like a flat block 
in Minecraft."}

calling tool: add_todo with args {"item":"If it is a rectangle or square, you can just multiply the two side 
lengths together."}

calling tool: add_todo with args {"item":"For a wobbling or tilted shape, pick one of the diagonals and break the 
quadrilateral into two triangles."}

calling tool: add_todo with args {"item":"Use the triangle area trick: take one side as the base, find a 
perpendicular height, and compute (1/2 × base × height)."}

...

Completed todo: Add the two triangle areas together to get the total area of the quadrilateral. - Notes: Add the 
two numbers together. The sum is the area of your whole quadrilateral shape—like the total surface you need tiles 
to cover in Minecraft.

calling tool: show_todos with args {}

0. Draw the quadrilateral on a big piece of paper so you can see all sides.
1. Check if your shape looks like a rectangle or a square—like a flat block in Minecraft.
2. If it is a rectangle or square, you can just multiply the two side lengths together.
3. For a wobbling or tilted shape, pick one of the diagonals and break the quadrilateral into two triangles.
4. Use the triangle area trick: take one side as the base, find a perpendicular height, and compute (1/2 × base × 
height).
5. Add the two triangle areas together to get the total area of the quadrilateral.

AI: Here’s the awesome plan you’ll use—each step is a little Minecraft‑building puzzle!

| Step | To‑Do | How it helps you find the area |
|------|-------|--------------------------------|
| 1 | **Draw the quadrilateral** | Make sure you can see all four corners—like sketching your own blocky house. |
| 2 | **Check if it’s a rectangle or square** | Two pairs of opposite sides look equal—just like a flat blocky 
room. |
| 3 | **If it’s a rectangle/square, multiply side1 × side2** | Simple multiplication gives you the flat area—just 
the ground you’d tile. |
| 4 | **For wobbly shapes, choose a diagonal and split into two triangles** | Think of it as building a diagonal 
bridge; each triangle shares that diagonal. |
| 5 | **Find each triangle’s area: pick a base, drop a perpendicular height, compute ½ × base × height** | The 
height is the shortest distance from the opposite corner to the base (“shadow” length). |
| 6 | **Add the two triangle areas together** | Happy‑math time! The sum is the full quadrilateral’s area—exactly 
how much floor you need. |
| 7 | **Celebrate** | You’ve mastered area‑finding, just like mastering creative mode building! 🎉 |

Now, you can follow these steps on any shape, and you’ll always know how big the inside is. Have fun measuring and 
building!

None
